## Extracting keywords

In [2]:
from keybert import KeyBERT

model_name = 'Qwen/Qwen3-Embedding-0.6B'
keybert = KeyBERT(model_name)

In [3]:
import pandas as pd

df = pd.read_csv('../../data/raw/hyperbook.csv', index_col='url')
sample = df.iloc[0]
content = sample['content']

In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(model_name='gpt-4', chunk_size=512, chunk_overlap=64)
chunker = lambda text: splitter.split_text(text)

splits = splitter.split_text(content)
keywords = keybert.extract_keywords(splits, top_n=10, nr_candidates=20)
flattened_keywords = [kw[0] for chunk_keywords in keywords for kw in chunk_keywords]

In [ ]:
from collections import Counter
from itertools import chain

def get_keywords(doc: str) -> list[str]:
    """
    Splits and embeds doc and returns the top 10
    most semantically relevant keywords
    """

    # Extract keywords over every chunk
    chunks = splitter.split_text(doc)
    result = keybert.extract_keywords(chunks, top_n=10)
    
    # Flatten, filter for strings and get the top n most common keywords
    all_keywords = [item for item in chain.from_iterable(result) if isinstance(item, str)]
    top_keywords = [pair[0] for pair in Counter(all_keywords).most_common(10)]
    
    return top_keywords


In [33]:
abstract = sample['abstract']

In [34]:
get_keywords(abstract)

['chlamydomonas',
 'chloroplasts',
 'organism',
 'algae',
 'prokaryotic',
 'eukaryotes',
 'organisms',
 'species',
 'originated',
 'genus']

In [15]:
get_keywords(content)

['chlamydomonas',
 'cilia',
 'phototaxis',
 'cilium',
 'ciliary',
 'algae',
 'photoreceptor',
 'flagella',
 'photoreceptors',
 'photoproteins']

In [9]:
from collections import Counter

Counter(flattened_keywords).most_common(10)

[('chlamydomonas', 13),
 ('flagella', 8),
 ('algae', 7),
 ('photoreceptor', 6),
 ('photosynthesis', 5),
 ('cilia', 5),
 ('phototaxis', 5),
 ('cilium', 4),
 ('ciliary', 4),
 ('photoreceptors', 4)]

In [6]:
sorted(kw_frequencies.items(), key=lambda item: item[1], reverse=True)[:10]

[('chlamydomonas', 13),
 ('flagella', 8),
 ('algae', 7),
 ('photoreceptor', 6),
 ('photosynthesis', 5),
 ('cilia', 5),
 ('phototaxis', 5),
 ('cilium', 4),
 ('ciliary', 4),
 ('photoreceptors', 4)]

In [7]:
keybert.extract_keywords(content, top_n=10, nr_candidates=20)

[('chlamydomonas', 0.627),
 ('chlorophytes', 0.613),
 ('ciliates', 0.6059),
 ('photopsin', 0.5924),
 ('photopsins', 0.5918),
 ('chloroplasts', 0.5802),
 ('chlorophyll', 0.5722),
 ('phototaxis', 0.5663),
 ('chloroplast', 0.5656),
 ('algae', 0.5645)]